# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed

    last_problem = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError) as e:
            last_problem = f'network error: {e!r}'
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                raise ValueError(f'HTTP {r.status_code}, not retrying: {r.text[:300]}')
            last_problem = f'HTTP {r.status_code}'

        if attempt < tries - 1:
            time.sleep(2 ** attempt)

    raise RuntimeError(f'get_weather failed after {tries} attempts (last: {last_problem})')
    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about

start, end = '2026-09-05', '2026-10-03'
wanted = ['2026-09-05', '2026-10-03']

used_fallback = False
try:
    raw = get_weather(start, end)
except RuntimeError as e:
    used_fallback = True
    print(f'WARNING: API failed, using fallback sample data. Reason: {e}')

weekends = raw[raw['time'].isin(wanted)].reset_index(drop=True)

if len(weekends) != len(wanted):
    raise ValueError(f'Expected {len(wanted)} rows, got {len(weekends)}: {weekends["time"].tolist()}')

print(f'used_fallback = {used_fallback}')
weekends

used_fallback = False

used_fallback = False


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

used_fallback = False
try:
    raw = get_weather('2026-09-05', '2026-10-03')
    wx = raw[raw['time'].isin(wanted)].reset_index(drop=True)
    if len(wx) != len(wanted):
        raise ValueError(f'Expected {len(wanted)} rows, got {wx["time"].tolist()}')
    wx['source'] = 'open-meteo'
except RuntimeError as e:
    used_fallback = True
    print(f'WARNING: API failed, using FALLBACK SAMPLE DATA. Reason: {e}')
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'

print(f'used_fallback = {used_fallback}')
wx

used_fallback = False


,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
clean = (
    wx.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm',
    })
    .assign(date=lambda d: pd.to_datetime(d['date'], format='%Y-%m-%d'))
    [['date', 'temp_max', 'precip_mm', 'source']]
    .sort_values('date')
    .reset_index(drop=True)
)

assert pd.api.types.is_datetime64_any_dtype(clean['date']), 'date is not a datetime'
assert clean['date'].is_unique, f'duplicate dates: {clean.loc[clean["date"].duplicated(), "date"].tolist()}'
assert clean[['date', 'temp_max', 'precip_mm']].notna().all().all(), 'nulls in key or value columns'
assert clean['source'].isin(['open-meteo', 'fallback sample']).all(), 'unexpected source label'

clean

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


In [6]:
wx = clean
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [7]:
a, b = wx.iloc[0], wx.iloc[1]
wetter, drier = (a, b) if a['precip_mm'] > b['precip_mm'] else (b, a)
diff = wetter['precip_mm'] - drier['precip_mm']

print(f"{wetter['date']:%B %d} was wetter: {wetter['precip_mm']:.1f} mm of rain "
      f"vs {drier['precip_mm']:.1f} mm on {drier['date']:%B %d}, "
      f"a difference of {diff:.1f} mm.")

October 03 was wetter: 14.7 mm of rain vs 2.5 mm on September 05, a difference of 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [8]:
# TODO
from datetime import datetime
from zoneinfo import ZoneInfo

LAT, LON = 38.03, -78.51
pulled_at = datetime.now(ZoneInfo('America/New_York'))

print('Source:      Open-Meteo Historical Weather API')
print('Endpoint:   ', BASE)
print('Coordinates:', f'{LAT}, {LON}')
print('Date range: ', f'{start} to {end} requested; kept {", ".join(wanted)}')
print('Pulled:     ', pulled_at.strftime('%Y-%m-%d %H:%M %Z'))
print('Data:       ', 'FALLBACK SAMPLE (API unreachable, not real observations)'
                      if used_fallback else 'real, from the API')

Source:      Open-Meteo Historical Weather API
Endpoint:    https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range:  2026-09-05 to 2026-10-03 requested; kept 2026-09-05, 2026-10-03
Pulled:      2026-10-11 20:32 EDT
Data:        real, from the API


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

There would be columns filled with NaN, which would then be skipped/ignored by matplotlib and pandas in later steps in data analysis. It would then be impossible to find any correlation between weather data and sales. The error wouldn't show because merging an empty DataFrame with a complete data frame isn't inheriently an error as long as the merge is done correctly, there would just be columns filled with NaN.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [9]:
# stretch
CITIES = {
    'Charlottesville': (38.03, -78.51),
    'Orlando':         (28.54, -81.38),
}

FALLBACKS = {
    'Charlottesville': FALLBACK,
    'Orlando': pd.DataFrame({
        'time': ['2026-09-05', '2026-10-03'],
        'temperature_2m_max': [32.0, 29.5],
        'precipitation_sum': [5.0, 0.0],
    }),
}

def city_weather(city, lat, lon):
    """One city's two dates, cleaned and labelled."""
    try:
        raw = get_weather('2026-09-05', '2026-10-03', lat=lat, lon=lon)
        out = raw[raw['time'].isin(wanted)].copy()
        if len(out) != len(wanted):
            raise ValueError(f'{city}: expected {len(wanted)} rows, got {out["time"].tolist()}')
        source = 'open-meteo'
    except RuntimeError as e:
        print(f'WARNING: {city} API failed, using FALLBACK SAMPLE. Reason: {e}')
        out = FALLBACKS[city].copy()
        source = 'fallback sample'

    return (out
        .rename(columns={'time': 'date',
                         'temperature_2m_max': 'temp_max',
                         'precipitation_sum': 'precip_mm'})
        .assign(date=lambda d: pd.to_datetime(d['date'], format='%Y-%m-%d'),
                city=city, source=source)
        [['city', 'date', 'temp_max', 'precip_mm', 'source']])

weather_long = pd.concat(
    [city_weather(c, lat, lon) for c, (lat, lon) in CITIES.items()],
    ignore_index=True,
)

# validate the long table before anything joins to it
assert list(weather_long.columns) == ['city', 'date', 'temp_max', 'precip_mm', 'source']
assert len(weather_long) == len(CITIES) * len(wanted), f'unexpected row count: {len(weather_long)}'
assert pd.api.types.is_datetime64_any_dtype(weather_long['date'])
assert not weather_long.duplicated(['city', 'date']).any(), 'duplicate (city, date) keys'
assert weather_long[['temp_max', 'precip_mm']].notna().all().all(), 'missing values'
assert (weather_long['precip_mm'] >= 0).all(), 'negative rainfall'

used_fallback = (weather_long['source'] == 'fallback sample').any()
print(f'used_fallback = {used_fallback}')
weather_long

used_fallback = False


,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
